In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc ffsim matplotlib numpy pyscf qiskit-addon-sqd
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# القطرنة الكمومية القائم على العينات لهاميلتوني كيمياء

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*تقدير الاستخدام: أقل من دقيقة واحدة على معالج Heron r2 (ملاحظة: هذا تقدير فقط. قد يختلف وقت تشغيلك الفعلي.)*
## نتائج التعلم
بعد الانتهاء من هذا البرنامج التعليمي، ينبغي أن يفهم المستخدمون:
- كيفية استخدام [إضافة SQD لـ Qiskit](https://github.com/Qiskit/qiskit-addon-sqd) لتقريب طاقة الحالة الأرضية لنظام جزيئي باستخدام سلاسل بت مأخوذة من وحدة معالجة كمومية (QPU).
- كيفية استخدام [ffsim](https://github.com/qiskit-community/ffsim) لبناء دائرة نموذج المجموعة العنقودية الوحدوية المحلية Jastrow (LUCJ) لمحاكاة الكيمياء الكمية.

## المتطلبات الأساسية
نوصي المستخدمين بالتعرف على المواضيع التالية قبل البدء بهذا البرنامج التعليمي:
- الكيمياء الكمية والتكميم الثاني
- استخدام الأولي Sampler لأخذ عينات من الدوائر الكمية

## الخلفية
في هذا البرنامج التعليمي، نوضح كيفية معالجة العينات الكمية الصاخبة لتقريب حالة الأرضية لجزيئة النيتروجين $\text{N}_2$ عند طول الرابطة في حالة الاتزان، وذلك باستخدام [إضافة SQD لـ Qiskit](https://github.com/Qiskit/qiskit-addon-sqd) لتنفيذ [خوارزمية القطرنة الكمومية القائم على العينات (SQD)](https://arxiv.org/abs/2405.05068). يمكن الاطلاع على مزيد من التفاصيل حول البرنامج في [التوثيق المقابل](/guides/qiskit-addons-sqd)، بما في ذلك [مثال بسيط](/guides/qiskit-addons-sqd-get-started) للبدء.

يُوصى بهذا البرنامج التعليمي للمستخدمين المألوفين بالكيمياء الكمية؛ وتحديدًا الإلمام بإيجاد طاقات الحالة الأرضية للجزيئات. للاطلاع على شرح تفصيلي لسير العمل، يمكن الرجوع إلى [دورة خوارزمية القطرنة الكمومية](https://quantum.cloud.ibm.com/learning/courses/quantum-diagonalization-algorithms).

SQD هي تقنية لإيجاد القيم الذاتية والمتجهات الذاتية للمؤثرات الكمية، مثل هاميلتوني النظام الكمي، باستخدام الحوسبة الكمية والحوسبة الكلاسيكية الموزعة معًا. تُستخدم الحوسبة الكلاسيكية الموزعة لمعالجة العينات المحصّلة من المعالج الكمي، وإسقاط هاميلتوني الهدف وقطرنةه في الفضاء الجزئي الذي تمتد إليه. تتضمن سير عمل SQD الخطوات التالية:

1.  اختيار نموذج الدائرة (ansatz) وتطبيقه على حاسوب كمي على حالة مرجعية (في هذه الحالة، حالة [هارتري-فوك](https://en.wikipedia.org/wiki/Hartree%E2%80%93Fock_method)).
2.  أخذ عينات من سلاسل البت الناتجة عن الحالة الكمية.
3.  تشغيل إجراء *استعادة التكوين ذاتي التناسق* على سلاسل البت للحصول على تقريب لحالة الأرضية.

يُعرف بأن SQD يعمل بشكل جيد عندما تكون الحالة الذاتية المستهدفة متفرقة: أي أن دالة الموجة تُدعم في مجموعة من الحالات الأساسية $\mathcal{S} = {|x\rangle }$ التي لا يتزايد حجمها أسيًا مع حجم المسألة.

### الكيمياء الكمية
يمكن كتابة هاميلتوني النظام الجزيئي على النحو التالي:

$$
\hat{H} = \sum_{ \substack{pr\\\sigma} } h_{pr} \, \hat{a}^\dagger_{p\sigma} \hat{a}_{r\sigma}
+ \frac12
\sum_{ \substack{prqs\\\sigma\tau} }
h_{prqs} \,
\hat{a}^\dagger_{p\sigma}
\hat{a}^\dagger_{q\tau}
\hat{a}_{s\tau}
\hat{a}_{r\sigma},
$$

حيث $h_{pr}$ و $h_{prqs}$ هي أعداد مركبة تُسمى التكاملات الجزيئية، ويمكن حسابها من مواصفات الجزيئة باستخدام برنامج حاسوبي. في هذا البرنامج التعليمي، نحسب هذه التكاملات باستخدام حزمة البرامج [PySCF](https://pyscf.org/).

للاطلاع على تفاصيل حول كيفية اشتقاق الهاميلتوني الجزيئي، يُرجى الرجوع إلى أحد كتب الكيمياء الكمية (على سبيل المثال، *Modern Quantum Chemistry* بقلم Szabo وOstlund). للحصول على شرح عالي المستوى حول كيفية تعيين مسائل الكيمياء الكمية على الحواسيب الكمية، اطلع على المحاضرة [*Mapping Problems to Qubits*](https://youtube.com/watch?v=TyFU6r8uEsE\&t=900) من Qiskit Global Summer School 2024.

### نموذج المجموعة العنقودية الوحدوية المحلية Jastrow (LUCJ)
تتطلب SQD نموذج دائرة كمية (ansatz) لأخذ العينات منه. في هذا البرنامج التعليمي، سنستخدم نموذج [المجموعة العنقودية الوحدوية المحلية Jastrow (LUCJ)](https://pubs.rsc.org/en/content/articlelanding/2023/sc/d3sc02516k) نظرًا لما يجمعه من التحفيز الفيزيائي والملاءمة للأجهزة. وسنستخدم [ffsim](https://qiskit-community.github.io/ffsim/) لبناء دائرة النموذج.

يتكيّف نموذج LUCJ مع وحدات معالجة الكيوبتات ذات الاتصالية المحدودة، إذ تُعيَّن مدارات السبين على كيوبتات بحيث لا يستلزم النموذج توجيهًا ببوابات SWAP. تمتلك أجهزة IBM&reg; طبولوجيا كيوبتات بشبكة سداسية ثقيلة (heavy-hex)، وفي هذه الحالة يمكننا اعتماد نمط "متعرج"، موضح أدناه. في هذا النمط، تُعيَّن المدارات ذات السبين نفسه على كيوبتات بطبولوجيا خطية (الدوائر الحمراء والزرقاء)، وتوجد وصلة بين مدارات ذات سبين مختلف عند كل رابع مدار مكاني، مع تيسير الوصلة بكيوبت إضافي (دوائر بنفسجية).

![مخطط تعيين الكيوبتات لنموذج LUCJ على شبكة heavy-hex](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/7e0ee7e1-2d24-417f-ac59-25c58db79aa9.avif)

### استعادة التكوين ذاتي التناسق
صُمِّم إجراء استعادة التكوين ذاتي التناسق لاستخلاص أكبر قدر ممكن من الإشارة من العينات الكمية الصاخبة. بما أن الهاميلتوني الجزيئي يحفظ عدد الجسيمات والزخم الزاوي الذاتي Z، فمن المنطقي اختيار نموذج دائرة يحفظ هذه التماثلات أيضًا. عند تطبيقه على حالة هارتري-فوك، تمتلك الحالة الناتجة عددًا ثابتًا من الجسيمات وزخمًا زاويًا ذاتيًا Z ثابتًا في الحالة الخالية من الضوضاء. لذلك، يجب أن يكون لنصفَي سبين-$\alpha$ وسبين-$\beta$ من أي سلسلة بت مأخوذة من هذه الحالة [وزن هامينغ](https://en.wikipedia.org/wiki/Hamming_weight) مماثل لما في حالة هارتري-فوك. بسبب وجود الضوضاء في المعالجات الكمية الحالية، ستنتهك بعض سلاسل البت المقيسة هذه الخاصية. أبسط صيغة للاختيار اللاحق هو تجاهل سلاسل البت هذه، لكن ذلك مُبذِّر لأن تلك السلاسل قد لا تزال تحتوي على بعض الإشارة. يحاول إجراء الاستعادة ذاتية التناسق استرداد بعض من تلك الإشارة في المعالجة اللاحقة. الإجراء تكراري ويتطلب كمدخلات تقديرًا لمتوسط إشغالات كل مدار في حالة الأرضية، والذي يُحسب أولًا من العينات الخام. يُشغَّل الإجراء في حلقة، وتشمل كل تكرارة الخطوات التالية:

1. لكل سلسلة بت تنتهك التماثلات المحددة، يتم قلب بتاتها بإجراء احتمالي مصمم لتقريب سلسلة البت من التقدير الحالي لمتوسط إشغالات المدارات، للحصول على سلسلة بت جديدة.
2. جمع كل سلاسل البت القديمة والجديدة التي تستوفي التماثلات، وأخذ عينات فرعية من مجموعات بحجم ثابت محدد مسبقًا.
3. لكل مجموعة فرعية من سلاسل البت، إسقاط الهاميلتوني في الفضاء الجزئي الممتد بالمتجهات الأساسية المقابلة (انظر [القسم السابق](#quantum-chemistry) للاطلاع على وصف هذه المتجهات الأساسية)، وحساب تقدير حالة الأرضية للهاميلتوني المُسقَط على حاسوب كلاسيكي.
4. تحديث تقدير متوسط إشغالات المدارات بتقدير حالة الأرضية ذي الطاقة الأدنى.

### مخطط سير عمل SQD
يُصوَّر سير عمل SQD في المخطط التالي:

![مخطط سير عمل خوارزمية SQD](../docs/images/tutorials/improving-energy-estimation-of-a-fermionic-hamiltonian-with-sqd/fd7e816f-4e2e-4dd7-a7da-f71afb9ca68d.avif)
## المتطلبات
قبل البدء في هذا البرنامج التعليمي، تأكد من تثبيت ما يلي:

*   Qiskit SDK الإصدار 1.0 أو أحدث، مع دعم [التصوير البياني](https://docs.quantum.ibm.com/api/qiskit/visualization)
*   Qiskit Runtime الإصدار 0.22 أو أحدث (`pip install qiskit-ibm-runtime`)
*   إضافة SQD لـ Qiskit الإصدار 0.11 أو أحدث (`pip install qiskit-addon-sqd`)
*   ffsim الإصدار 0.0.75 أو أحدث (`pip install ffsim`)
## الإعداد

In [1]:
import math

import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

## مثال المحاكي على نطاق صغير

في هذا البرنامج التعليمي، سنجد تقريبًا لحالة الأرضية لجزيئة النيتروجين بالقرب من مسافة الرابطة في حالة الاتزان. نستخدم أولًا مجموعة أساسية صغيرة STO-6G كي نتمكن من محاكاة التجربة والتحقق من صحتها.

### الخطوة 1: تعيين المدخلات الكلاسيكية إلى مسألة كمية

أولًا، نُحدد الجزيئة وخصائصها.

In [2]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="sto-6g",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -108.464957764796
CASCI E = -108.595987350986  E(CI) = -32.4115475088426  S^2 = 0.0000000
norb = 8
nelec = (5, 5)


Before constructing the LUCJ ansatz circuit, we first perform a CCSD calculation in the following code cell. The [$t_1$ and $t_2$ amplitudes](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) from this calculation will be used to initialize the parameters of the ansatz.

In [3]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -108.5933309085008  E_corr = -0.1283731437052354


قبل بناء دائرة نموذج LUCJ، نجري أولًا حساب CCSD في خلية الكود التالية. ستُستخدَم [سعات $t_1$ و$t_2$](https://en.wikipedia.org/wiki/Coupled_cluster#Cluster_operator) من هذا الحساب لتهيئة معاملات النموذج.

In [4]:
import warnings

from qiskit.transpiler import CouplingMap

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
coupling_map = CouplingMap.from_heavy_hex(3)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()

### Step 2: Optimize for quantum hardware execution

Next, we optimize the circuit for a target hardware. Typically, this step involves initializing the hardware backend and a pass manager for that backend. However, since the LUCJ ansatz is adapted to the hardware connectivity, we already performed these actions in the previous step. All that is left to do is run the pass manager on the circuit to transpile it to an ISA circuit that can be directly executed on the QPU.

In [5]:
isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")

Gate counts: OrderedDict({'xx_plus_yy': 86, 'p': 16, 'measure': 16, 'cp': 15, 'x': 10, 'swap': 2, 'barrier': 1})


الآن، نستخدم [ffsim](https://github.com/qiskit-community/ffsim) لإنشاء دائرة النموذج. بما أن جزيئتنا لها حالة هارتري-فوك ذات قشرة مغلقة، نستخدم المتغير المتوازن للسبين من نموذج UCJ، [UCJOpSpinBalanced](https://qiskit-community.github.io/ffsim/api/ffsim.html#ffsim.UCJOpSpinBalanced). نضع `optimize=True` في طريقة `from_t_amplitudes` لتمكين التحليل العاملي المزدوج "المضغوط" لسعات $t_2$ (انظر [نموذج المجموعة العنقودية الوحدوية المحلية Jastrow (LUCJ)](https://qiskit-community.github.io/ffsim/explanations/lucj.html#Parameter-initialization-from-CCSD) في توثيق ffsim للتفاصيل).

بما أن نموذج LUCJ يتكيّف مع الاتصالية المتاحة في وحدة معالجة الكيوبتات، يجب تهيئة الواجهة الخلفية لوحدة معالجة الكيوبتات قبل إنشاء النموذج. سننشئ الآن واجهة خلفية عامة بخريطة اقتران heavy-hex ومجموعة بوابات تتحلل إليها طبيعيًا دائرة نموذج LUCJ. ثم سنستخدم `ffsim.qiskit.generate_lucj_pass_manager` لإنشاء مدير تمرير متخصص في تحويل نموذج LUCJ إلى الواجهة الخلفية المعطاة وفق تخطيط "zig-zag" الموصوف في [قسم الخلفية حول نموذج LUCJ](#local-unitary-cluster-jastrow-lucj-ansatz). تستخدم هذه الدالة إرشادًا للتسجيل لتقليل الأخطاء المرتبطة بالتخطيط المختار، وهو أمر مهم عند استخدام وحدة معالجة كيوبتات حقيقية أو محاكٍ بنموذج ضوضاء. إضافةً إلى إعادة مدير التمرير، تُعيد هذه الدالة أيضًا أزواج الاقتران بين مدارات alpha-beta التي يمكن تنفيذها على الجهاز. إذا لم تكن جميع الأزواج قابلة للتنفيذ، تُصدر تحذيرًا.

In [6]:
rng = np.random.default_rng()
sampler = StatevectorSampler(seed=rng)
job = sampler.run([isa_circuit], shots=100_000)

In [7]:
primitive_result = job.result()
pub_result = primitive_result[0]

### Step 4: Post-process and return result in desired classical format

A useful metric to judge the quality of the QPU output is the number of valid configurations returned. A valid configuration has the correct particle number and spin Z, which means that the right half of the bitstring has Hamming weight equal to the number of spin-up electrons, and the left half has Hamming weight equal to the number of spin-down electrons. The following cell computes the fraction of sampled configurations that are valid.

In [8]:
def is_valid_bitstring(
    bitstring: str, norb: int, nelec: tuple[int, int]
) -> bool:
    n_alpha, n_beta = nelec
    return (
        len(bitstring) == 2 * norb
        and bitstring[norb:].count("1") == n_alpha
        and bitstring[:norb].count("1") == n_beta
    )


bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")

Fraction of sampled configurations that are valid: 1.0


### الخطوة 3: التنفيذ باستخدام الأوليات الكمية في Qiskit
بعد تحسين الدائرة للتنفيذ على الجهاز، أصبحنا جاهزين لتشغيلها على الجهاز الهدف وجمع العينات لتقدير طاقة حالة الأرضية. بما أن لدينا دائرة واحدة فقط، سنستخدم [وضع تنفيذ المهمة](/guides/execution-modes) في Qiskit Runtime وتنفيذ دائرتنا.

In [9]:
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)

Expected fraction of valid configurations from uniformly random bitstrings: 0.0478515625


Now, we estimate the ground state energy of the Hamiltonian using the `diagonalize_fermionic_hamiltonian` function. This function performs the self-consistent configuration recovery procedure to iteratively refine the noisy quantum samples to improve the energy estimate. We pass a callback function so that we can save the intermediate results for later analysis. See the [API documentation](/docs/api/qiskit-addon-sqd/fermion#diagonalize_fermionic_hamiltonian) for explanations of the arguments to `diagonalize_fermionic_hamiltonian`.

Here, we use the `initial_occupancies` argument to `diagonalize_fermionic_hamiltonian` to specify the Hartree-Fock configuration as the initial guess for the orbital occupancies in the ground state. This approach is sensible for systems where the ground state has significant support on the Hartree-Fock configuration, but it might not be appropriate in other situations, though more advanced computational methods might yield better initial guesses in those cases. Specifying `initial_occupancies` also allows configuration recovery to run even if no valid configurations were sampled, as may be the case when sampling a large circuit on a noisy QPU. Without this argument, the configuration recovery would fail and raise an error if no valid configurations were provided.

In [10]:
from functools import partial

from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
    solve_sci_batch,
)

# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(
            f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}"
        )


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

Iteration 1
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Iteration 2
	Subsample 0
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 1
		Energy: -108.59275573641656
		Subspace dimension: 900
	Subsample 2
		Energy: -108.59275573641656
		Subspace dimension: 900
Final energy: -108.59275573641656
Final energy error: 0.0032316145694579745


#### Visualize the results

The first plot shows that in this simulation we are already within `1 mH` of the exact answer after the first iteration (chemical accuracy is typically accepted to be `1 kcal/mol` $\approx$ `1.6 mH`). This is a small system, though, and because the samples are noiseless, configuration recovery is not needed. On a larger system run on a noisy QPU, multiple configuration recovery iterations might be needed, and the final accuracy might be worse. Generally, the energy can be improved by allowing more configuration recovery iterations or by increasing the number of samples per batch.

The second plot shows the average occupancy of each spatial orbital after the final iteration. We can see that both the spin-up and spin-down electrons occupy the first five orbitals with high probability in our solutions.

In [11]:
# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/caffd888-e89c-4aa9-8bae-4d1bb723b35e-0.avif" alt="Output of the previous code cell" />

### الخطوة 4: المعالجة اللاحقة وإعادة النتيجة بالصيغة الكلاسيكية المطلوبة
مقياس مفيد للحكم على جودة مخرجات وحدة معالجة الكيوبتات هو عدد التكوينات الصالحة المُعادة. التكوين الصالح هو الذي يمتلك عدد الجسيمات وزخم السبين Z الصحيحَين، أي أن النصف الأيمن من سلسلة البت له وزن هامينغ مساوٍ لعدد الإلكترونات ذات السبين الصاعد، والنصف الأيسر له وزن هامينغ مساوٍ لعدد الإلكترونات ذات السبين الهابط. تحسب الخلية التالية نسبة التكوينات المأخوذة عينةً منها التي تعدّ صالحة.

In [12]:
# ------------------------------ Step 1 ------------------------------
# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (1.0, 0, 0)]],
    basis="cc-pvdz",
    symmetry="Dooh",
)

# Define active space
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Store reference energy from SCI calculation performed separately
reference_energy = -109.22802921665716

print(f"norb = {norb}")
print(f"nelec = {nelec}")

# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(f"Using backend {backend.name}")

# Create pass manager
pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
    backend=backend,
    norb=norb,
    connectivity="heavy-hex",
    interaction_pairs=(pairs_aa, pairs_ab),
    optimization_level=3,
)

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
circuit.measure_all()


# ------------------------------ Step 2 ------------------------------

isa_circuit = pass_manager.run(circuit)
print(f"Gate counts: {isa_circuit.count_ops()}")


# ------------------------------ Step 3 ------------------------------
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["TUT_SQD"]
job = sampler.run([isa_circuit], shots=100_000)
primitive_result = job.result()
pub_result = primitive_result[0]


# ------------------------------ Step 4 ------------------------------

bit_array = pub_result.data.meas
num_valid = sum(
    is_valid_bitstring(b, norb, nelec) for b in bit_array.get_bitstrings()
)
valid_fraction = num_valid / bit_array.num_shots
print(f"Fraction of sampled configurations that are valid: {valid_fraction}")
expected_fraction_random = (
    math.comb(norb, n_alpha) * math.comb(norb, n_beta) / 2 ** (2 * norb)
)
print(
    f"Expected fraction of valid configurations from uniformly random bitstrings: {expected_fraction_random}"
)
# SQD options
energy_tol = 1e-3
occupancies_tol = 1e-3
max_iterations = 5

# Eigenstate solver options
num_batches = 3
samples_per_batch = 300
symmetrize_spin = True
carryover_threshold = 1e-4
max_cycle = 200

# Use the Hartree-Fock configuration as an initial guess for the orbital occupancies
initial_occupancies = (
    np.array([1] * n_alpha + [0] * (norb - n_alpha)),
    np.array([1] * n_beta + [0] * (norb - n_beta)),
)

# Pass options to the built-in eigensolver. If you just want to use the defaults,
# you can omit this step, in which case you would not specify the sci_solver argument
# in the call to diagonalize_fermionic_hamiltonian below.
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# List to capture intermediate results
result_history = []


result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=norb,
    nelec=nelec,
    num_batches=num_batches,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    max_iterations=max_iterations,
    sci_solver=sci_solver,
    symmetrize_spin=symmetrize_spin,
    initial_occupancies=initial_occupancies,
    carryover_threshold=carryover_threshold,
    callback=callback,
    seed=rng,
)

final_energy = result.energy + nuclear_repulsion_energy
energy_error = final_energy - reference_energy
print(f"Final energy: {final_energy}")
print(f"Final energy error: {energy_error}")

# Data for energies plot
x1 = range(len(result_history))
min_e = [
    min(result, key=lambda res: res.energy).energy + nuclear_repulsion_energy
    for result in result_history
]
e_diff = [abs(e - reference_energy) for e in min_e]
yt1 = [1.0, 1e-1, 1e-2, 1e-3, 1e-4]

# Chemical accuracy (+/- 1 milli-Hartree)
chem_accuracy = 0.001

# Data for avg spatial orbital occupancy
y2 = np.sum(result.orbital_occupancies, axis=0)
x2 = range(len(y2))

fig, axs = plt.subplots(1, 2, figsize=(12, 6))

# Plot energies
axs[0].plot(x1, e_diff, label="energy error", marker="o")
axs[0].set_xticks(x1)
axs[0].set_xticklabels(x1)
axs[0].set_yticks(yt1)
axs[0].set_yticklabels(yt1)
axs[0].set_yscale("log")
axs[0].set_ylim(1e-4)
axs[0].axhline(
    y=chem_accuracy,
    color="#BF5700",
    linestyle="--",
    label="chemical accuracy",
)
axs[0].set_title("Approximated Ground State Energy Error vs SQD Iterations")
axs[0].set_xlabel("Iteration Index", fontdict={"fontsize": 12})
axs[0].set_ylabel("Energy Error (Ha)", fontdict={"fontsize": 12})
axs[0].legend()

# Plot orbital occupancy
axs[1].bar(x2, y2, width=0.8)
axs[1].set_xticks(x2)
axs[1].set_xticklabels(x2)
axs[1].set_title("Avg Occupancy per Spatial Orbital")
axs[1].set_xlabel("Orbital Index", fontdict={"fontsize": 12})
axs[1].set_ylabel("Avg Occupancy", fontdict={"fontsize": 12})

plt.tight_layout()
plt.show()

converged SCF energy = -108.929838385609
norb = 26
nelec = (5, 5)
E(CCSD) = -109.2177884185544  E_corr = -0.2879500329450045
Using backend ibm_boston


Removing interaction (24, 24) from the end.
Removing interaction (20, 20) from the end.


Gate counts: OrderedDict({'sx': 7039, 'rz': 6990, 'cz': 1858, 'x': 61, 'measure': 52, 'barrier': 1})
Fraction of sampled configurations that are valid: 0.02124
Expected fraction of valid configurations from uniformly random bitstrings: 9.607888706852918e-07
Iteration 1
	Subsample 0
		Energy: -109.13889134249762
		Subspace dimension: 120409
	Subsample 1
		Energy: -109.11785470455858
		Subspace dimension: 110889
	Subsample 2
		Energy: -109.13234360554011
		Subspace dimension: 130321
Iteration 2
	Subsample 0
		Energy: -109.16392179579177
		Subspace dimension: 223729
	Subsample 1
		Energy: -109.16281938332986
		Subspace dimension: 223729
	Subsample 2
		Energy: -109.16955816711932
		Subspace dimension: 233289
Iteration 3
	Subsample 0
		Energy: -109.17905772999075
		Subspace dimension: 324900
	Subsample 1
		Energy: -109.17532445048462
		Subspace dimension: 357604
	Subsample 2
		Energy: -109.1733168689756
		Subspace dimension: 348100
Iteration 4
	Subsample 0
		Energy: -109.18437778820451
		Su

<Image src="../docs/images/tutorials/sample-based-quantum-diagonalization/extracted-outputs/3858949c-a55d-4ff8-a0fc-54fb53e131b5-3.avif" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
If you found this work interesting, you might be interested in the following material:
- [Sample-based Krylov quantum diagonalization of a fermionic lattice model](/docs/tutorials/sample-based-krylov-quantum-diagonalization) - a related tutorial using time evolution circuits instead of a variational ansatz
- [Scale SQD chemistry workflows with Dice solver](https://qiskit.github.io/qiskit-addon-sqd/how_tos/integrate_dice_solver.html) - a page showing how to use the more efficient Dice software for diagonalization
- [SQD addon API documentation](https://qiskit.github.io/qiskit-addon-sqd/apidocs/qiskit_addon_sqd.fermion.html#qiskit_addon_sqd.fermion.diagonalize_fermionic_hamiltonian) - reference for the `diagonalize_fermionic_hamiltonian` function
- [*Chemistry beyond the scale of exact diagonalization on a quantum-centric supercomputer*](https://www.science.org/doi/10.1126/sciadv.adu9991) - the paper this tutorial is based on
</Admonition>